## Basic Analog CiM Macro

Welcome to the CiMLoop Tutorials. This first tutorial will guide you through the
process of creating a basic analog CiM macro. Using this macro, we will explore
the CiMLoop syntax and important CiM components.

### Introduction and Overview

The diagram below shows the structure of the analog CiM macro we will be
creating. In the diagram, we show a 3x3 weight matrix and a 3x3 array for
illustrative purposes, but in simulation we'll be using a 32x32 array.

![Analog CiM Macro](images/how_cim_works.svg)

This analog CiM macro stores a weight matrix and performs matrix-vector
operations using analog computation. For simplicity, in this tutorial we assume
that analog components are high-enough resolution to process all operands.

The macro consists of three main components:

- Memory Array: A 2D array of memory elements that store the weight matrix. Each
  of these memory elements, which we call a CiM unit, stores a weight and
  performs an analog multiply-accumulate (MAC) operation between its stored
  weight and an analog input value.
- Row Drivers + DAC: These are the circuits that provide analog input values to
  the memory array. The Digital Analog Converter (DACs) convert digital input
  values to analog voltages. The row drivers then apply these voltages to the
  rows of the memory array. *In this tutorial, we'll be using a simple
  pulse-train DAC that is folded into the row drivers. We'll therefore only be
  seeing a row driver in the macro definition.*
- Column Drivers + ADC: These are the circuits that read out the analog output
  values from the memory array. Column drivers select and read of CiM array
  columns, and Analog Digital Converters (ADCs) convert analog outputs from each
  column into digital values.

We'll dive into the details of each of these components later on, but for now,
we'll start by looking at the structure and syntax of how to define this macro
in CiMLoop.

## Container-Hierarchy Representation

CiMLoop uses a container-hierarchy representation to define the structure of CiM macros.
A **container** is a grouping of components and/or (sub)containers, and a
**container-hierarchy** is a series of containers where each contains all subsequent
components/containers. Expressing the CiM macro as a hierarchy lets us decompose the
macro into smaller, easier-to-understand components. It also lets the Timeloop mapper
map to a wide variety of CiM macros.

Using a container-hierarchy representation, we can break down our CiM array into three
levels.

### The Macro Level 

At the top level, we have the macro. The macro contains the ADC, column drivers, and row
drivers (note that the DAC is included in the row drivers).

Below the macro, we break the CiM array into column containers, where each container
represents one column of CiM units. Row drivers supply input values onto the rows; each
row connects to all columns, so these input values are *reused* between columns. Column
drivers read outputs from each column, and an ADC converts the analog outputs into
digital values.

Our three-column CiM array has three column containers.

### The Column Level

Looking inside each column, we see that there is no additional hardware; instead, the
column is immediately broken down further into multiple rows. Each row receives a
different input value, and the analog outputs are reused (*i.e.*, summed on a wire)
between multiple rows.

### The Row Level

Finally, looking inside each row, we see one `CimUnit` that stores a weight value. In
this macro, each `CimUnit` is capable of performing an analog MAC operation between an
8b input and an 8b weight. To represent this operation, we use 64x 1b MAC units, which
act as a virtual representation of the computation that occurs within the `CimUnit`.

In [ ]:
from _scripts import *

display_important_variables("basic_analog")
get_spec("basic_analog").arch

The YAML file of the macro is commented to show important attributes.

In [ ]:
txt = open(af.examples.arches.compute_in_memory.basic_analog).read()
display(
    Markdown(
        f"""
```yaml
{txt}
```
        """
    )
)

## Running the Accelerator

### Basic Run

To run the accelerator, we'll load in the spec using one of our pre-defined helper
functions get_spec. We can use the workload and renames from the `basic_analog`
example, which are written to fully utilize the CiM array.

The display_dnn_results function will display energy and latency of each Einsum, as well
as give energy and latency breakdowns.


We can make several observations from the results:

- The CiM array is fully utilized.
- The ADC and Row Drivers dominate the energy and latency of the macro.
- We can directly see the number of ADC and row driver (DAC) uses per MAC operation in
  the actions table at the bottom. Both of these come out to $1/32$ in this test, which
  reflects that the array is getting full reuse.

In [ ]:
basic_analog = af.examples.arches.compute_in_memory.basic_analog
workload = af.Workload.from_yaml(basic_analog, top_key="workload")
renames = af.Renames.from_yaml(basic_analog, top_key="renames")

spec = get_spec("basic_analog", add_dummy_main_memory=True, n_macros=1)
spec.workload = workload
spec.workload.einsums = spec.workload.einsums
spec.renames = renames
spec.mapper.metrics = af.Metrics.ENERGY

# It's OK to explore loop orders, but will make the search take longer, and is not very
# helpful for CiM accelerators. It's not very helpful because loop orders help us
# capture sliding window reuse, and the CiM macros consume very big tiles at once,
# giving limited benefit from the small overlap between each tile.
spec.mapper.explore_loop_orders = False

result = spec.map_workload_to_arch()
result = result.drop_components_with_zero_energy_and_latency()

display_dnn_results(result, title="Basic Analog Results")

We can visualilze the full mapping as a LoopTree by rendering the mapping result.

In [ ]:
result

Looking at the above, notice that it's fully utilizing the $32\times32$ CiM array by
mapping the $K$ and $N$ ranks across the rows and columns of the array, respectively.

To find out why the number of rows and columns influence energy efficiency,
we'll plot the energy breakdown of the CiM array as we vary the number of rows,
the number of columns, and then both at the same time.

We can see the following:

1. In all tests, the `cim_unit`s consume and the `row_drivers` consume little
   overall energy, while the `adc` dominates overall energy.
2. As the number of rows increases, `adc` energy per MAC decreases.
3. As the number of columns increases, `row_drivers` energy per MAC decreases.

These observations show the hints at one of the key design considerations in CiM
macros: how to reduce the area and energy cost of column readout circuitry, in
particular the ADCs. As we can see here, using more rows in the CiM array can
help reduce the energy cost of column readout circuits. This is just one of many
possible ways to reduce column readout energy. We'll explore other possibilities
later on.

Of course, this is just one macro; many other macros pay more energy for
`cim_unit` and `row_drivers` than for `adc`. There is a rich design
space to explore!

Before moving on, it may be helpful to stop for a moment and consider the
following questions:

1. Why does `adc` energy per compute decrease as the number of rows
   increase, but not the number of columns? And vice versa, why does
   `row_drivers` energy per compute decrease as the number of columns
   decreases, but not the number of rows?
2. Could there be other ways to decrease ADC, DAC, and row/column driver energy?

In [ ]:
def _run_basic_analog_array_size(rows, cols):
    spec = get_spec("basic_analog", add_dummy_main_memory=True, n_macros=1)
    spec.arch.find_spatial("row_ARRAY_ROWS").fanout = rows
    spec.arch.find_spatial("column_ARRAY_COLUMNS").fanout = cols
    spec.workload.einsums["Matmul"].rank_sizes["K"] = rows
    spec.workload.einsums["Matmul"].rank_sizes["N"] = cols
    result = spec.map_workload_to_arch()
    result = result.drop_components_with_zero_energy_and_latency()
    return rows, cols, result


row_choices = [32, 64, 128, 256]
col_choices = [32, 64, 128, 256]
results = parallel(
    [
        delayed(_run_basic_analog_array_size)(r, c)
        for r in row_choices
        for c in col_choices
    ]
)

In [ ]:
# labels = [f"{r} rows, {c} cols" for r, c, _ in results]
# energies = [r.energy(per_component=True) for _, _, r in results]

to_plot = {}
for rows, cols, result in results:
    for k, v in result.per_compute().energy(per_component=True).items():
        label = f"{rows} rows, {cols} cols"
        to_plot.setdefault(k, {})[label] = v

# Transpose it for the plotting function

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
bar_comparison(
    to_plot,
    "Array Size",
    "Energy (J)/Compute",
    "Energy Breakdown by Array Size",
    ax,
)
plt.tight_layout()